<a href="https://colab.research.google.com/github/omaradelahmed/fly-rank-internship1/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omaradelahmed/fly-rank-internship1/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*
**Lane 2: Refresh / Content Opportunity Scoring.**

I'm choosing this lane over freestyling. The starter dataset already ships with a clear,
measurable outcome (`trend_direction` / `is_declining_label`) that flags content whose organic
search impressions dropped more than 20% over the last 30 days versus the prior 30 days. In the
30,000-row starter sample, 54.2% of content is currently in this declining state — that's not an
edge case, it's the dominant pattern across all 32 clients. A problem this common, with a
measurable outcome already sitting in the data, is a strong candidate for a repeatable
decision-support tool rather than a one-off analysis.

## 2. The question: decision, action, cost of a wrong call

**Decision:** Which content pages should be prioritized for a refresh this cycle?

**Who acts, and how:**
- Content editors use the model's output to decide which pages to work on first.
- Account managers use the same signal to proactively flag at-risk pages to clients, before the
  client notices the drop themselves.

**Task type:** Binary classification — will this page's impressions show a declining trend
(>20% drop over the last 30 days vs. the prior 30 days), or not?

**Cost of a wrong call:**
- *False positive* (flagged as declining, but it wasn't): wasted editor hours reviewing or
  refreshing a page that didn't actually need it.
- *False negative* (missed a real decline): the page keeps losing traffic silently until someone
  notices manually — a real, compounding cost, since organic traffic is slow to recover once lost.

**Why ML, and not a simple rule:** A single-variable threshold doesn't cleanly separate the two
groups. For example, average `engagement_rate` is only slightly lower for declining pages (2.44%)
than for stable/growing ones (2.65%) — not a clean cutoff. The real signal is likely a combination
of several weaker features (content age, word count, position, click behavior, etc.), which is
exactly the kind of tangled, multi-signal pattern ML is suited for, and a plain if-statement is
not.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")


Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [4]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Total rows:", len(df))
print("Unique clients:", df["client_id"].nunique())
print()

# trend_direction is the source of is_declining_label (never used as a feature — see Section 2)
print(df["trend_direction"].value_counts())
print()

declining_rate = (df["trend_direction"] == "down").mean()
print(f"Share of content currently declining (>20% drop, 30d vs prev 30d): {declining_rate:.1%}")

df["declining"] = df["trend_direction"].eq("down")
eng_by_group = df.groupby("declining")["engagement_rate"].mean()
print()
print("Average engagement_rate by group (declining vs. not):")
print(eng_by_group)

Total rows: 30000
Unique clients: 32

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Share of content currently declining (>20% drop, 30d vs prev 30d): 54.2%

Average engagement_rate by group (declining vs. not):
declining
False    2.649728
True     2.437193
Name: engagement_rate, dtype: float64


## 4. Careful words: what I can and can't claim

**What this work can claim:**
- Observed, historical patterns in the last 90 days of pseudonymized content performance across
  32 clients.
- A directional, decision-support signal: pages flagged as high-risk resemble the ~54% of content
  that historically went on to decline — not a certainty for any single page.
- Correlational relationships between page-level features (engagement, content age, position,
  etc.) and an observed drop in impressions.

**What this work can't claim:**
- Causal proof that refreshing a page will reverse a decline — the data shows association, not
  the effect of an intervention.
- "Predicting Google" or reverse-engineering the search algorithm — the model scores pages against
  a historical pattern, it does not model the ranking system itself.
- Guaranteed accuracy for any individual client or page — the panel is uneven (different clients,
  content types, and volumes), so the model's output should be read as a prioritization signal,
  not a verdict.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.